# Probability: Calculus Meets Randomness

## Why Probability Matters for Calculus

Probability and calculus are **deeply connected**:

- **Integration**: Computing probabilities requires integration!
- **Differentiation**: Finding modes and deriving distributions
- **Expected Values**: Integration of probability distributions
- **Transformations**: Change of variables uses the Jacobian
- **Limit Theorems**: Central Limit Theorem uses limits

**The Big Idea**: Continuous probability is all about calculus - PDFs are integrated to get probabilities, CDFs are differentiated to get PDFs!

---

## Learning Objectives

By the end of this notebook, you will:
- Master probability fundamentals
- Work with discrete and continuous distributions
- Use integration to compute probabilities
- Apply calculus to probability problems
- Understand the Central Limit Theorem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

# Set random seed for reproducibility
np.random.seed(42)

## 1. Probability Fundamentals

### Sample Space and Events

- **Sample Space** $\Omega$: Set of all possible outcomes
- **Event** $A$: Subset of sample space
- **Probability** $P(A)$: Number between 0 and 1

### Axioms of Probability

1. $0 \leq P(A) \leq 1$ for any event $A$
2. $P(\Omega) = 1$ (something must happen)
3. If $A$ and $B$ are mutually exclusive: $P(A \cup B) = P(A) + P(B)$

### Conditional Probability

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

### Bayes' Theorem (Fundamental!)

$$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$$

In [ ]:
# Bayes' Theorem Example: Medical Test
print("=" * 70)
print("BAYES' THEOREM EXAMPLE: Medical Testing")
print("=" * 70)

# Problem: Disease affects 1% of population
# Test is 95% accurate (both sensitivity and specificity)
# If you test positive, what's the probability you have the disease?

P_disease = 0.01  # Prior probability
P_pos_given_disease = 0.95  # Sensitivity (true positive rate)
P_pos_given_healthy = 0.05  # False positive rate (1 - specificity)
P_healthy = 1 - P_disease

# Total probability of positive test
P_pos = P_pos_given_disease * P_disease + P_pos_given_healthy * P_healthy

# Bayes' theorem: P(disease | positive test)
P_disease_given_pos = (P_pos_given_disease * P_disease) / P_pos

print(f"\nGiven:")
print(f"  Disease prevalence: {P_disease*100:.1f}%")
print(f"  Test sensitivity: {P_pos_given_disease*100:.1f}%")
print(f"  Test specificity: {(1-P_pos_given_healthy)*100:.1f}%")
print(f"\nCalculation:")
print(f"  P(positive test) = {P_pos:.4f}")
print(f"\nUsing Bayes' Theorem:")
print(f"  P(disease | positive) = {P_disease_given_pos:.4f} = {P_disease_given_pos*100:.2f}%")
print(f"\nSurprising Result: Even with a 95% accurate test,")
print(f"a positive result only means {P_disease_given_pos*100:.1f}% chance of disease!")
print(f"This is because the disease is rare (1% prevalence).")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Tree diagram
ax1.text(0.1, 0.9, 'Population', fontsize=12, fontweight='bold', transform=ax1.transAxes)
ax1.text(0.1, 0.75, f'Disease: {P_disease*100:.1f}%', fontsize=10, transform=ax1.transAxes, color='red')
ax1.text(0.1, 0.65, f'  → Test Pos: {P_pos_given_disease*100:.0f}%', fontsize=9, transform=ax1.transAxes)
ax1.text(0.1, 0.55, f'  → Test Neg: {(1-P_pos_given_disease)*100:.0f}%', fontsize=9, transform=ax1.transAxes)
ax1.text(0.1, 0.40, f'Healthy: {P_healthy*100:.1f}%', fontsize=10, transform=ax1.transAxes, color='green')
ax1.text(0.1, 0.30, f'  → Test Pos: {P_pos_given_healthy*100:.0f}%', fontsize=9, transform=ax1.transAxes)
ax1.text(0.1, 0.20, f'  → Test Neg: {(1-P_pos_given_healthy)*100:.0f}%', fontsize=9, transform=ax1.transAxes)
ax1.text(0.1, 0.05, f'If positive: P(disease) = {P_disease_given_pos*100:.1f}%', 
         fontsize=11, fontweight='bold', transform=ax1.transAxes,
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
ax1.axis('off')
ax1.set_title('Probability Tree', fontsize=14, fontweight='bold')

# Bar chart comparison
categories = ['Prior\n(Before Test)', 'Posterior\n(After Positive)']
probs = [P_disease * 100, P_disease_given_pos * 100]
ax2.bar(categories, probs, color=['blue', 'red'], alpha=0.7)
ax2.set_ylabel('Probability of Disease (%)', fontsize=12)
ax2.set_title('How Test Changes Our Belief', fontsize=14, fontweight='bold')
ax2.set_ylim(0, 25)
for i, (cat, prob) in enumerate(zip(categories, probs)):
    ax2.text(i, prob + 1, f'{prob:.1f}%', ha='center', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Discrete Random Variables

### Probability Mass Function (PMF)

For discrete $X$:
$$P(X = x) = p(x)$$

Properties:
- $p(x) \geq 0$ for all $x$
- $\sum_{all\ x} p(x) = 1$

### Expected Value (Mean)

$$E[X] = \sum_{all\ x} x \cdot p(x)$$

### Variance

$$Var(X) = E[(X - \mu)^2] = E[X^2] - (E[X])^2$$

### Common Discrete Distributions

- **Binomial**: $n$ trials, probability $p$ of success
- **Poisson**: Count of rare events
- **Geometric**: Trials until first success

In [ ]:
# Discrete distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Binomial distribution
n, p = 20, 0.3
x_binom = np.arange(0, n+1)
pmf_binom = stats.binom.pmf(x_binom, n, p)
axes[0, 0].bar(x_binom, pmf_binom, alpha=0.7, color='blue', edgecolor='black')
axes[0, 0].set_xlabel('Number of Successes', fontsize=11)
axes[0, 0].set_ylabel('Probability', fontsize=11)
axes[0, 0].set_title(f'Binomial(n={n}, p={p})\nE[X] = {n*p:.1f}, Var(X) = {n*p*(1-p):.2f}', 
                     fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Poisson distribution  
lam = 4
x_pois = np.arange(0, 15)
pmf_pois = stats.poisson.pmf(x_pois, lam)
axes[0, 1].bar(x_pois, pmf_pois, alpha=0.7, color='green', edgecolor='black')
axes[0, 1].set_xlabel('Count', fontsize=11)
axes[0, 1].set_ylabel('Probability', fontsize=11)
axes[0, 1].set_title(f'Poisson(λ={lam})\nE[X] = {lam}, Var(X) = {lam}', 
                     fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Geometric distribution
p_geom = 0.2
x_geom = np.arange(1, 20)
pmf_geom = stats.geom.pmf(x_geom, p_geom)
axes[1, 0].bar(x_geom, pmf_geom, alpha=0.7, color='red', edgecolor='black')
axes[1, 0].set_xlabel('Trial of First Success', fontsize=11)
axes[1, 0].set_ylabel('Probability', fontsize=11)
axes[1, 0].set_title(f'Geometric(p={p_geom})\nE[X] = {1/p_geom:.1f}, Var(X) = {(1-p_geom)/p_geom**2:.2f}', 
                     fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Comparison of means and variances
axes[1, 1].axis('off')
summary_text = f"""DISCRETE DISTRIBUTIONS SUMMARY

Binomial(n={n}, p={p}):
  • Models: n independent trials with prob p
  • E[X] = np = {n*p:.1f}
  • Var(X) = np(1-p) = {n*p*(1-p):.2f}
  • Example: 20 coin flips, P(heads) = 0.3

Poisson(λ={lam}):
  • Models: Rate of rare events  
  • E[X] = λ = {lam}
  • Var(X) = λ = {lam}
  • Example: Customers per hour = 4

Geometric(p={p_geom}):
  • Models: Trials until first success
  • E[X] = 1/p = {1/p_geom:.1f}
  • Var(X) = (1-p)/p² = {(1-p_geom)/p_geom**2:.2f}
  • Example: Rolls until first 6 (p=1/6)

Key Property: ∑ P(X=x) = 1 (all probabilities sum to 1)
"""
axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
               fontsize=10, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plt.show()

print("Expected Value Calculation Example (Binomial):")
print(f"  E[X] = ∑ x·P(X=x) = {sum(x_binom * pmf_binom):.3f}")
print(f"  Formula: E[X] = np = {n}·{p} = {n*p}")

## 3. Continuous Random Variables: WHERE CALCULUS COMES IN!

### Probability Density Function (PDF)

For continuous $X$:
$$f(x) \geq 0 \text{ for all } x$$
$$\int_{-\infty}^{\infty} f(x) dx = 1$$

**CALCULUS**: Probability is the integral!
$$P(a \leq X \leq b) = \int_a^b f(x) dx$$

### Cumulative Distribution Function (CDF)

$$F(x) = P(X \leq x) = \int_{-\infty}^x f(t) dt$$

**CALCULUS**: PDF is the derivative of CDF!
$$f(x) = \frac{d}{dx} F(x)$$

### Expected Value (Uses Integration!)

$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) dx$$

### Variance

$$Var(X) = \int_{-\infty}^{\infty} (x - \mu)^2 f(x) dx = E[X^2] - (E[X])^2$$

In [ ]:
# PDF vs CDF relationship - CALCULUS IN ACTION!
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Normal distribution
mu, sigma = 0, 1
x = np.linspace(-4, 4, 1000)
pdf = stats.norm.pdf(x, mu, sigma)
cdf = stats.norm.cdf(x, mu, sigma)

# Plot 1: PDF
axes[0, 0].plot(x, pdf, 'b-', linewidth=3, label='PDF: f(x)')
axes[0, 0].fill_between(x, pdf, alpha=0.3)
axes[0, 0].set_xlabel('x', fontsize=12)
axes[0, 0].set_ylabel('Density', fontsize=12)
axes[0, 0].set_title('Probability Density Function (PDF)\n∫f(x)dx = 1', 
                     fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Plot 2: CDF  
axes[0, 1].plot(x, cdf, 'r-', linewidth=3, label='CDF: F(x)')
axes[0, 1].axhline(y=0.5, color='g', linestyle='--', alpha=0.5, label='Median')
axes[0, 1].set_xlabel('x', fontsize=12)
axes[0, 1].set_ylabel('Probability', fontsize=12)
axes[0, 1].set_title('Cumulative Distribution Function\nF(x) = ∫₋∞ˣ f(t)dt', 
                     fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Plot 3: Computing P(a ≤ X ≤ b) by integration
a, b = -1, 1.5
x_fill = x[(x >= a) & (x <= b)]
pdf_fill = pdf[(x >= a) & (x <= b)]

axes[1, 0].plot(x, pdf, 'b-', linewidth=3)
axes[1, 0].fill_between(x_fill, pdf_fill, alpha=0.5, color='red', 
                        label=f'P({a} ≤ X ≤ {b})')
axes[1, 0].axvline(x=a, color='green', linestyle='--', linewidth=2)
axes[1, 0].axvline(x=b, color='green', linestyle='--', linewidth=2)

# Calculate probability using integration
prob = stats.norm.cdf(b, mu, sigma) - stats.norm.cdf(a, mu, sigma)
axes[1, 0].text(0, 0.3, f'∫₍₋₁₎¹·⁵ f(x)dx = {prob:.4f}', fontsize=12, ha='center',
               bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

axes[1, 0].set_xlabel('x', fontsize=12)
axes[1, 0].set_ylabel('Density', fontsize=12)
axes[1, 0].set_title('Probability by Integration\nP(a ≤ X ≤ b) = ∫ₐᵇ f(x)dx', 
                     fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Plot 4: Relationship between PDF and CDF
axes[1, 1].plot(x, pdf, 'b-', linewidth=3, label='PDF = F\' (x)', alpha=0.7)
axes[1, 1].plot(x, cdf, 'r-', linewidth=3, label='CDF = ∫ f(t)dt', alpha=0.7)

# Show derivative relationship at a point
x_pt = 1
pdf_pt = stats.norm.pdf(x_pt, mu, sigma)
cdf_pt = stats.norm.cdf(x_pt, mu, sigma)
axes[1, 1].plot(x_pt, pdf_pt, 'bo', markersize=12, label=f'f({x_pt}) = {pdf_pt:.3f}')
axes[1, 1].plot(x_pt, cdf_pt, 'ro', markersize=12, label=f'F({x_pt}) = {cdf_pt:.3f}')

axes[1, 1].set_xlabel('x', fontsize=12)
axes[1, 1].set_ylabel('Value', fontsize=12)
axes[1, 1].set_title('CALCULUS: f(x) = dF/dx\nPDF is derivative of CDF!', 
                     fontsize=13, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("="*70)
print("CALCULUS IN PROBABILITY")
print("="*70)
print(f"\n1. Computing P(-1 ≤ X ≤ 1.5) by INTEGRATION:")
print(f"   ∫₋₁¹·⁵ f(x)dx = {prob:.4f}")
print(f"\n2. Using CDF (which is an integral):")
print(f"   F(1.5) - F(-1) = {stats.norm.cdf(1.5):.4f} - {stats.norm.cdf(-1):.4f} = {prob:.4f}")
print(f"\n3. PDF as DERIVATIVE of CDF:")
print(f"   f({x_pt}) = F'({x_pt}) ≈ {pdf_pt:.4f}")
print(f"\nThis is why calculus is essential for continuous probability!")

## 4. Important Continuous Distributions

### Uniform Distribution

$$f(x) = \begin{cases} \frac{1}{b-a} & a \leq x \leq b \\ 0 & \text{otherwise} \end{cases}$$

### Exponential Distribution (Waiting times)

$$f(x) = \lambda e^{-\lambda x}, \quad x \geq 0$$

$$E[X] = \frac{1}{\lambda}, \quad Var(X) = \frac{1}{\lambda^2}$$

### Normal (Gaussian) Distribution

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

**Most important distribution!**
- Central Limit Theorem
- Appears everywhere in nature

In [ ]:
# Major continuous distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Uniform distribution
a_unif, b_unif = 2, 5
x_unif = np.linspace(0, 7, 1000)
pdf_unif = stats.uniform.pdf(x_unif, a_unif, b_unif - a_unif)
axes[0, 0].plot(x_unif, pdf_unif, 'b-', linewidth=3)
axes[0, 0].fill_between(x_unif, pdf_unif, alpha=0.3)
axes[0, 0].set_xlabel('x', fontsize=11)
axes[0, 0].set_ylabel('Density', fontsize=11)
axes[0, 0].set_title(f'Uniform({a_unif}, {b_unif})\nE[X] = {(a_unif+b_unif)/2:.1f}, height = {1/(b_unif-a_unif):.3f}', 
                     fontsize=13, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_ylim(0, 0.5)

# Exponential distribution
lam_exp = 0.5
x_exp = np.linspace(0, 10, 1000)
pdf_exp = stats.expon.pdf(x_exp, scale=1/lam_exp)
axes[0, 1].plot(x_exp, pdf_exp, 'r-', linewidth=3)
axes[0, 1].fill_between(x_exp, pdf_exp, alpha=0.3, color='red')
axes[0, 1].axvline(x=1/lam_exp, color='green', linestyle='--', linewidth=2, label=f'Mean = {1/lam_exp:.1f}')
axes[0, 1].set_xlabel('x', fontsize=11)
axes[0, 1].set_ylabel('Density', fontsize=11)
axes[0, 1].set_title(f'Exponential(λ={lam_exp})\nE[X] = 1/λ = {1/lam_exp:.1f}', 
                     fontsize=13, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Normal distribution - multiple parameters
x_norm = np.linspace(-8, 8, 1000)
params = [(0, 1, 'blue'), (0, 2, 'red'), (2, 1, 'green')]
for mu_n, sigma_n, color in params:
    pdf_norm = stats.norm.pdf(x_norm, mu_n, sigma_n)
    axes[1, 0].plot(x_norm, pdf_norm, color=color, linewidth=2.5, 
                   label=f'μ={mu_n}, σ={sigma_n}')
axes[1, 0].set_xlabel('x', fontsize=11)
axes[1, 0].set_ylabel('Density', fontsize=11)
axes[1, 0].set_title('Normal Distribution Family\nf(x) = (1/σ√2π)e^(-(x-μ)²/2σ²)', 
                     fontsize=13, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Integration example: Computing E[X] for exponential
axes[1, 1].axis('off')
calc_text = f"""EXPECTED VALUE BY INTEGRATION

For Exponential(λ={lam_exp}):

E[X] = ∫₀^∞ x · f(x) dx
     = ∫₀^∞ x · λe^(-λx) dx
     
Using integration by parts:
  u = x,  dv = λe^(-λx)dx
  du = dx,  v = -e^(-λx)
  
E[X] = [-xe^(-λx)]₀^∞ + ∫₀^∞ e^(-λx)dx
     = 0 + [-1/λ · e^(-λx)]₀^∞
     = 0 - (-1/λ)
     = 1/λ
     
For λ = {lam_exp}:
  E[X] = 1/{lam_exp} = {1/lam_exp:.1f}

Variance (similar integration):
  Var(X) = E[X²] - (E[X])²
         = 2/λ² - 1/λ²  
         = 1/λ²
         = 1/{lam_exp}² = {1/lam_exp**2:.1f}
"""
axes[1, 1].text(0.05, 0.95, calc_text, transform=axes[1, 1].transAxes,
               fontsize=9, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.3))

plt.tight_layout()
plt.show()

# Numerical verification
print("Numerical Verification of E[X] for Exponential:")
result, error = quad(lambda x: x * lam_exp * np.exp(-lam_exp * x), 0, np.inf)
print(f"  ∫₀^∞ x·λe^(-λx)dx = {result:.4f}")
print(f"  Formula 1/λ = {1/lam_exp}")
print(f"  Match: {np.isclose(result, 1/lam_exp)}")

## 5. Central Limit Theorem

**THE MOST IMPORTANT THEOREM IN STATISTICS!**

If $X_1, X_2, ..., X_n$ are i.i.d. random variables with mean $\mu$ and variance $\sigma^2$, then:

$$\frac{\bar{X} - \mu}{\sigma/\sqrt{n}} \xrightarrow{d} N(0, 1) \text{ as } n \to \infty$$

**In words**: The sample mean approaches a normal distribution, regardless of the original distribution!

**Why it matters**:
- Justifies using normal distribution in many applications
- Foundation of statistical inference
- Works even for non-normal populations!

In [ ]:
# Central Limit Theorem demonstration
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

# Start with a very non-normal distribution (exponential)
lam_clt = 2
sample_sizes = [1, 5, 30]  # n values
n_samples = 10000

# True population parameters
pop_mean = 1/lam_clt
pop_std = 1/lam_clt

for col, n in enumerate(sample_sizes):
    # Generate sample means
    sample_means = []
    for _ in range(n_samples):
        sample = np.random.exponential(1/lam_clt, n)
        sample_means.append(np.mean(sample))
    sample_means = np.array(sample_means)
    
    # Row 1: Original distribution (for n=1 it's just the population)
    if n == 1:
        axes[0, col].hist(sample_means, bins=50, density=True, alpha=0.7, color='red', edgecolor='black')
        x_pop = np.linspace(0, 3, 1000)
        axes[0, col].plot(x_pop, stats.expon.pdf(x_pop, scale=1/lam_clt), 'b-', linewidth=3, label='True PDF')
    else:
        axes[0, col].hist(sample_means, bins=50, density=True, alpha=0.7, color='orange', edgecolor='black')
    
    axes[0, col].set_title(f'Sample Size n = {n}', fontsize=13, fontweight='bold')
    axes[0, col].set_ylabel('Density', fontsize=11)
    axes[0, col].grid(True, alpha=0.3)
    if n == 1:
        axes[0, col].legend()
    
    # Row 2: Distribution of sample means
    axes[1, col].hist(sample_means, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black',
                     label=f'Sample means (n={n})')
    
    # Overlay theoretical normal distribution from CLT
    x_norm = np.linspace(sample_means.min(), sample_means.max(), 1000)
    theoretical_std = pop_std / np.sqrt(n)
    pdf_norm = stats.norm.pdf(x_norm, pop_mean, theoretical_std)
    axes[1, col].plot(x_norm, pdf_norm, 'r-', linewidth=3, label=f'N({pop_mean:.2f}, {theoretical_std:.3f}²)')
    
    axes[1, col].axvline(x=pop_mean, color='green', linestyle='--', linewidth=2, label=f'μ = {pop_mean:.2f}')
    axes[1, col].set_ylabel('Density', fontsize=11)
    axes[1, col].grid(True, alpha=0.3)
    axes[1, col].legend(fontsize=9)
    
    # Row 3: Standardized (Z-scores)
    z_scores = (sample_means - pop_mean) / (pop_std / np.sqrt(n))
    axes[2, col].hist(z_scores, bins=50, density=True, alpha=0.7, color='lightgreen', edgecolor='black',
                     label='Standardized')
    
    # Standard normal overlay
    x_std = np.linspace(-4, 4, 1000)
    pdf_std = stats.norm.pdf(x_std, 0, 1)
    axes[2, col].plot(x_std, pdf_std, 'r-', linewidth=3, label='N(0,1)')
    
    axes[2, col].set_xlabel('Standardized Value', fontsize=11)
    axes[2, col].set_ylabel('Density', fontsize=11)
    axes[2, col].set_xlim(-4, 4)
    axes[2, col].grid(True, alpha=0.3)
    axes[2, col].legend(fontsize=9)

# Add row labels
fig.text(0.01, 0.83, 'Population\n(Exponential)', ha='center', va='center', fontsize=12, fontweight='bold', rotation=90)
fig.text(0.01, 0.50, 'Sample\nMeans', ha='center', va='center', fontsize=12, fontweight='bold', rotation=90)
fig.text(0.01, 0.17, 'Standardized\n(Z-scores)', ha='center', va='center', fontsize=12, fontweight='bold', rotation=90)

plt.suptitle('Central Limit Theorem in Action\nStarting from Exponential → Sample Means Become Normal!', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0.02, 0, 1, 0.99])
plt.show()

print("="*70)
print("CENTRAL LIMIT THEOREM VERIFICATION")
print("="*70)
print(f"\nOriginal Distribution: Exponential(λ={lam_clt})")
print(f"  Population μ = {pop_mean:.3f}")
print(f"  Population σ = {pop_std:.3f}")
print(f"\nSample Means Distribution:")
for n in sample_sizes:
    sample_means_test = [np.mean(np.random.exponential(1/lam_clt, n)) for _ in range(n_samples)]
    print(f"\n  n = {n}:")
    print(f"    Sample mean of means: {np.mean(sample_means_test):.4f} (should be ≈ {pop_mean:.3f})")
    print(f"    Sample std of means:  {np.std(sample_means_test):.4f} (should be ≈ {pop_std/np.sqrt(n):.3f})")
    print(f"    Theoretical std:      σ/√n = {pop_std:.3f}/√{n} = {pop_std/np.sqrt(n):.4f}")

print(f"\nKey Insight: As n increases, distribution of sample means becomes normal!")
print(f"This works even though we started with a highly skewed exponential distribution.")

## 6. Key Takeaways

### Probability Fundamentals
- **Bayes' Theorem**: Update beliefs with new evidence
- **Conditional Probability**: P(A|B) = P(A∩B)/P(B)

### Discrete vs Continuous
- **Discrete**: PMF, sum probabilities
- **Continuous**: PDF, **integrate** for probabilities

### CALCULUS CONNECTIONS (CRITICAL!)
- **Probability**: P(a ≤ X ≤ b) = **∫ₐᵇ f(x)dx** (integration!)
- **CDF**: F(x) = **∫₋∞ˣ f(t)dt** (integration!)
- **PDF from CDF**: f(x) = **dF/dx** (differentiation!)
- **Expected Value**: E[X] = **∫ x·f(x)dx** (integration!)
- **Variance**: Var(X) = **∫ (x-μ)²f(x)dx** (integration!)

### Important Distributions
- **Binomial**: Discrete, n trials
- **Poisson**: Discrete, rare events  
- **Exponential**: Continuous, waiting times
- **Normal**: Continuous, appears everywhere (CLT!)

### Central Limit Theorem
- Sample means → Normal distribution (as n → ∞)
- Works regardless of original distribution
- Foundation of statistical inference

---

**Next Up**: Statistics - using probability and calculus for inference and decision-making!